In [1]:
!pip install -q torch>=2.0 scikit-learn pandas numpy scipy lightgbm


In [2]:
# --- Module: data.py ---
import torch
import torch.nn as nn
from torch.autograd import Function
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import StandardScaler, LabelEncoder
from pathlib import Path
import warnings
import json
import os
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

# Columns to log-scale (large numeric values)
# Columns to log-scale (large, positive-skew numeric values)
_LOG_SCALE_CANDIDATES = [
    # Memory / buffer sizes (both legacy + underscore variants)
    "buffer_pool_size", "cache_effective_size", "cache_effective_size_",
    "per_query_memory", "maintenance_memory", "temp_memory",
    "log_buffer_size", "log_capacity", "temp_file_limit_",
    
    # Resource / capacity knobs that can span wide ranges
    "max_connections_", "io_parallelism",
    
    # Workload counters
    "blks_hit", "blks_read",
    "disk_read_bytes", "disk_read_count",
    "disk_write_bytes", "disk_write_count",
    "tup_fetched", "tup_returned", "tup_inserted", "tup_updated", "tup_deleted",
    "xact_commit", "xact_rollback",
    "conflicts",
    
    # Optional timing/size-like knobs (keep if present)
    "commit_delay_",
    "vacuum_cost_limit_", "vacuum_cost_page_dirty_", "vacuum_cost_page_hit_", "vacuum_cost_page_miss_",
]

LABEL_COL = "cost"
DOMAIN_COL = "domain_id"
QP_EMB_COL = "qp_emb_vector"
SOURCE_DOMAIN = 3
META_COLS = ["db_engine", "hardware", "ram_gb"]


def get_log_scale_cols(df):
    return [c for c in _LOG_SCALE_CANDIDATES if c in df.columns]


def get_mask_cols(df):
    return [c for c in df.columns if c.startswith("mask_")]


def get_feature_cols(df, mask_cols):
    exclude = set(mask_cols + [LABEL_COL, DOMAIN_COL, QP_EMB_COL] + META_COLS)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    return [c for c in numeric_cols if c not in exclude]

def expand_qp_emb(df, col=None):
    """Expand qp_emb_vector list column into per-dimension float columns"""
    if col is None:
        col = QP_EMB_COL
    emb_matrix = np.vstack(df[col].values)
    qp_cols = [f"qp_emb_{i}" for i in range(emb_matrix.shape[1])]
    emb_df = pd.DataFrame(emb_matrix, columns=qp_cols, index=df.index)
    return emb_df, qp_cols


def preprocess(da, normalization_mode="per_engine", log_target=False):
    """
    Complete preprocessing pipeline:
    1. Convert embeddings from JSON strings
    2. Log-scale large numeric features
    3. Fill NaN values
    4. Filter zero-variance features
    5. Expand query plan embeddings
    6. Fit categorical encoders and feature scaler
    7. NORMALIZE COSTS with selected mode: per_engine or global
    
    Returns: processed df, feature columns, encoders, scaler, and normalization params
    """
    df = da.copy()

    # Convert qp_emb_vector from JSON strings if needed
    if df[QP_EMB_COL].dtype == object and isinstance(df[QP_EMB_COL].iloc[0], str):
        df[QP_EMB_COL] = df[QP_EMB_COL].apply(
            lambda x: json.loads(x) if isinstance(x, str) else x
        )
        print("INFO: Converted qp_emb_vector from JSON strings")

    log_scale_cols = get_log_scale_cols(df)
    if log_scale_cols:
        print(f"INFO: Log-scaling {len(log_scale_cols)} feature columns")
    # Log-scale large numeric features
    for col in log_scale_cols:
        if col in df.columns:
            df[col] = np.log1p(df[col].clip(lower=0))

    mask_cols = get_mask_cols(df)
    feat_cols = get_feature_cols(df, mask_cols)

    # Replace non-finite feature/mask values, then fill NaNs
    df[feat_cols] = df[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    df[mask_cols] = df[mask_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    # Filter zero-variance features
    feat_cols_filtered = [c for c in feat_cols if df[c].var() > 1e-10]
    dropped_zero_var = [c for c in feat_cols if c not in feat_cols_filtered]
    if dropped_zero_var:
        print(f"WARN: Dropped {len(dropped_zero_var)} zero-variance features: {dropped_zero_var[:5]}")

    # Clip extreme values to prevent scaling instability
    for col in feat_cols_filtered:
        df[col] = np.clip(df[col], -1e4, 1e4)

    # Expand query plan embeddings and concatenate
    emb_df, qp_cols = expand_qp_emb(df)
    emb_df = emb_df.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    df = pd.concat([df.reset_index(drop=True), emb_df.reset_index(drop=True)], axis=1)
    print(f"INFO: Expanded {len(qp_cols)} query plan embedding dimensions")

    # Clean cost column BEFORE computing normalization stats
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
    bad_cost_mask = ~np.isfinite(df[LABEL_COL].to_numpy())
    bad_cost_count = int(bad_cost_mask.sum())
    if bad_cost_count > 0:
        df = df.loc[~bad_cost_mask].reset_index(drop=True)
        print(f"WARN: Dropped {bad_cost_count} rows with non-finite cost values before normalization")

    if len(df) == 0:
        raise ValueError("No rows left after removing non-finite cost values.")

    # Fit categorical encoders
    db_enc = LabelEncoder().fit(df["db_engine"])
    hw_enc = LabelEncoder().fit(df["hardware"])
    print(f"INFO: Fitted encoders: {len(db_enc.classes_)} DB engines, {len(hw_enc.classes_)} hardware configs")

    # Fit feature scaler on SOURCE domain only (no target leakage)
    src_mask = df[DOMAIN_COL] == SOURCE_DOMAIN
    if src_mask.sum() > 0:
        fit_data = df.loc[src_mask, feat_cols_filtered].values
        print(f"INFO: Fitting scaler on {src_mask.sum()} source domain samples")
    else:
        fit_data = df[feat_cols_filtered].values
        print("WARN: No source domain samples, fitting scaler on all data")
    feat_scaler = StandardScaler().fit(fit_data)
    
    # Save the true raw cost prior to transform/normalization for MAPE/RMSE logging
    df["raw_cost"] = df[LABEL_COL].values

    # Optional target transform before normalization
    target_transform = {"type": "none"}
    if log_target:
        df[LABEL_COL] = np.log1p(np.maximum(df[LABEL_COL].values, 0.0))
        target_transform = {"type": "log1p"}

    # Cost normalization setup (finite-safe)
    global_mu = float(df[LABEL_COL].mean())
    global_sigma = float(df[LABEL_COL].std())
    if (not np.isfinite(global_mu)) or (not np.isfinite(global_sigma)) or (global_sigma < 1e-8):
        finite_cost = df[LABEL_COL].to_numpy()
        finite_cost = finite_cost[np.isfinite(finite_cost)]
        if finite_cost.size == 0:
            raise ValueError("Could not compute finite cost normalization stats.")
        global_mu = float(np.mean(finite_cost))
        global_sigma = float(np.std(finite_cost, ddof=1)) if finite_cost.size > 1 else 1.0
        if (not np.isfinite(global_sigma)) or (global_sigma < 1e-8):
            global_sigma = 1.0

    normalization_mode = normalization_mode.lower().strip()
    if normalization_mode == "global":
        cost_mu_map = {"__global__": global_mu}
        cost_sigma_map = {"__global__": global_sigma}
        print("\nINFO: Cost normalization mode: GLOBAL")
        print(f"   mu={global_mu:.4f}, sigma={global_sigma:.4f}")
    elif normalization_mode == "per_engine":
        stats_by_engine = df.groupby("db_engine")[LABEL_COL].agg(["mean", "std"])
        stats_by_engine["mean"] = stats_by_engine["mean"].replace([np.inf, -np.inf], np.nan).fillna(global_mu)
        stats_by_engine["std"] = stats_by_engine["std"].replace([np.inf, -np.inf], np.nan).fillna(global_sigma).clip(lower=1e-8)
        cost_mu_map = {k: float(v) for k, v in stats_by_engine["mean"].to_dict().items()}
        cost_sigma_map = {k: float(v) for k, v in stats_by_engine["std"].to_dict().items()}

        print("\nINFO: Cost normalization mode: PER_ENGINE")
        for engine in sorted(cost_mu_map.keys()):
            cnt = int((df["db_engine"] == engine).sum())
            print(f"   {engine}: n={cnt}, mu={cost_mu_map[engine]:.4f}, sigma={cost_sigma_map[engine]:.4f}")
    else:
        raise ValueError("normalization_mode must be 'global' or 'per_engine'")

    # Apply normalization
    if "__global__" in cost_mu_map:
        mu_series = pd.Series(cost_mu_map["__global__"], index=df.index)
        sigma_series = pd.Series(cost_sigma_map["__global__"], index=df.index).clip(lower=1e-8)
    else:
        mu_series = df["db_engine"].map(cost_mu_map).fillna(global_mu)
        sigma_series = df["db_engine"].map(cost_sigma_map).fillna(global_sigma).clip(lower=1e-8)
    df[LABEL_COL] = (df[LABEL_COL] - mu_series) / sigma_series

    # Final guard: remove any non-finite normalized targets
    bad_norm_mask = ~np.isfinite(df[LABEL_COL].to_numpy())
    bad_norm_count = int(bad_norm_mask.sum())
    if bad_norm_count > 0:
        df = df.loc[~bad_norm_mask].reset_index(drop=True)
        print(f"WARN: Dropped {bad_norm_count} rows with non-finite normalized costs")

    print(f"   Normalized overall: mu={df[LABEL_COL].mean():.4f}, sigma={df[LABEL_COL].std():.4f}")
    print(f"   Range: [{df[LABEL_COL].min():.4f}, {df[LABEL_COL].max():.4f}]")

    # Verification: normalized stats by engine family and domain groups
    mysql_mask = df[DOMAIN_COL].isin([0, 1])
    postgres_mask = df[DOMAIN_COL].isin([2, 3])
    if mysql_mask.any():
        print(
            f"   Normalized MySQL domains (0,1): mu={df.loc[mysql_mask, LABEL_COL].mean():.4f}, "
            f"sigma={df.loc[mysql_mask, LABEL_COL].std():.4f}"
        )
    if postgres_mask.any():
        print(
            f"   Normalized PostgreSQL domains (2,3): mu={df.loc[postgres_mask, LABEL_COL].mean():.4f}, "
            f"sigma={df.loc[postgres_mask, LABEL_COL].std():.4f}"
        )

    print("   Per-engine normalized mean/std:")
    print(df.groupby("db_engine")[LABEL_COL].agg(["mean", "std"]).round(4))

    return (
        df, feat_cols_filtered, mask_cols, qp_cols, db_enc, hw_enc, feat_scaler,
        cost_mu_map, cost_sigma_map, target_transform
    )

print("INFO: Preprocessing functions defined")

class CostModelDataset(Dataset):
    """
    PyTorch dataset for cost model training.
    
    IMPORTANT: Costs should be ALREADY NORMALIZED in preprocessing.
    This class just packages data for PyTorch - no transformation applied.
    """
    def __init__(self, df, feat_cols, mask_cols, qp_cols,
                 db_enc, hw_enc, feat_scaler,
                 labeled_target_idx=None, raw_costs=None):
        # Feature matrix: [scaled features | masks | qp embeddings]
        X_feat = feat_scaler.transform(df[feat_cols].values.astype(np.float32))
        X_mask = df[mask_cols].values.astype(np.float32)
        X_qp = df[qp_cols].values.astype(np.float32)
        self.x_feat = np.concatenate([X_feat, X_mask, X_qp], axis=1)

        # Conditioning inputs for regression head
        self.db_oh = np.eye(len(db_enc.classes_), dtype=np.float32)[
                         db_enc.transform(df["db_engine"])]
        self.hw_oh = np.eye(len(hw_enc.classes_), dtype=np.float32)[
                         hw_enc.transform(df["hardware"])]
        self.ram = df["ram_gb"].values.astype(np.float32).reshape(-1, 1)

        # Labels and domain IDs (costs already normalized)
        self.cost = df[LABEL_COL].values.astype(np.float32)
        self.domain = df[DOMAIN_COL].values.astype(np.int64)
        
        # Raw costs for unnormalized metric evaluation during training
        if raw_costs is not None:
            self.raw_cost = raw_costs.astype(np.float32)
        else:
            self.raw_cost = self.cost

        # has_label: 1 = use this row in task loss
        self.has_label = (self.domain == SOURCE_DOMAIN).astype(np.float32)
        if labeled_target_idx is not None:
            self.has_label[list(labeled_target_idx)] = 1.0

    def __len__(self):
        return len(self.x_feat)

    def __getitem__(self, idx):
        return {
            "x_feat": torch.tensor(self.x_feat[idx]),
            "db_oh": torch.tensor(self.db_oh[idx]),
            "hw_oh": torch.tensor(self.hw_oh[idx]),
            "ram": torch.tensor(self.ram[idx]),
            "cost": torch.tensor(self.cost[idx]),
            "raw_cost": torch.tensor(self.raw_cost[idx]),
            "domain": torch.tensor(self.domain[idx]),
            "has_label": torch.tensor(self.has_label[idx]),
        }


def build_sampler(dataset):
    """
    WeightedRandomSampler for balanced domain batches.
    Prevents source domain (largest) from dominating every batch.
    """
    domains = dataset.domain
    class_counts = np.bincount(domains, minlength=4).astype(np.float32)
    weights = 1.0 / class_counts[domains]
    return WeightedRandomSampler(weights, num_samples=len(dataset), replacement=True)

print("INFO: Dataset and sampler defined")




INFO: Preprocessing functions defined
INFO: Dataset and sampler defined


In [3]:
# --- Module: models.py ---
import torch
import torch.nn as nn
from torch.autograd import Function
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from pathlib import Path
import warnings
import json
import os
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

class GRLFunction(Function):
    """Gradient Reversal Layer - reverses gradients during backprop"""
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None


class GRL(nn.Module):
    """Gradient Reversal Layer module"""
    def __init__(self, lambda_=1.0):
        super().__init__()
        self.lambda_ = lambda_

    def set_lambda(self, lambda_):
        self.lambda_ = lambda_

    def forward(self, x):
        return GRLFunction.apply(x, self.lambda_)

print("INFO: GRL defined")

class SAINTBlock(nn.Module):
    """Single SAINT transformer block with feature and row attention"""
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.feat_attn = nn.MultiheadAttention(d_model, n_heads,
                                                dropout=dropout, batch_first=True)
        self.row_attn = nn.MultiheadAttention(d_model, n_heads,
                                               dropout=dropout, batch_first=True)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ln3 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # Feature attention (within sample, across features)
        res = x
        x2, _ = self.feat_attn(x, x, x)
        x = self.ln1(res + x2)

        # Row attention (within feature, across batch)
        xt = x.transpose(0, 1)
        res = xt
        xt2, _ = self.row_attn(xt, xt, xt)
        xt = self.ln2(res + xt2)
        x = xt.transpose(0, 1)

        # Feed-forward
        res = x
        x = self.ln3(res + self.ff(x))
        return x


class SAINTEncoder(nn.Module):
    """Gf: tabular features -> latent vector z"""
    def __init__(self, num_features, d_model=128, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.num_features = num_features
        self.feat_embed = nn.Embedding(num_features, d_model)
        self.val_proj = nn.Linear(1, d_model)
        self.blocks = nn.ModuleList([SAINTBlock(d_model, n_heads, dropout)
                                     for _ in range(n_layers)])
        self.ln_out = nn.LayerNorm(d_model)

    def forward(self, x):
        B, F = x.shape
        feat_ids = torch.arange(F, device=x.device).unsqueeze(0).expand(B, -1)
        tokens = self.feat_embed(feat_ids) + self.val_proj(x.unsqueeze(-1))
        for block in self.blocks:
            tokens = block(tokens)
        z = self.ln_out(tokens).mean(dim=1)
        return z

print("INFO: SAINT encoder defined")

class RegressionHead(nn.Module):
    """Gy: [z | db_engine_oh | hardware_oh | ram_gb] -> predicted cost"""
    def __init__(self, z_dim, n_db_engines, n_hardware, dropout=0.1):
        super().__init__()
        cond_dim = z_dim + n_db_engines + n_hardware + 1
        self.net = nn.Sequential(
            nn.Linear(cond_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, z, db_engine_oh, hardware_oh, ram_gb):
        x = torch.cat([z, db_engine_oh, hardware_oh, ram_gb], dim=1)
        return self.net(x).squeeze(1)


class DomainClassifier(nn.Module):
    """Gd: z -> domain_id logits (4 classes)"""
    def __init__(self, z_dim, n_domains=4, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, n_domains),
        )

    def forward(self, z):
        return self.net(z)

print("INFO: Regression head and domain classifier defined")

class DANNCostModel(nn.Module):
    """Complete DANN model with SAINT encoder, regression head, and domain classifier"""
    def __init__(self, num_features, n_db_engines, n_hardware,
                 d_model=128, n_heads=4, n_layers=2, n_domains=4, dropout=0.1):
        super().__init__()
        self.Gf = SAINTEncoder(num_features, d_model, n_heads, n_layers, dropout)
        self.Gy = RegressionHead(d_model, n_db_engines, n_hardware, dropout)
        self.Gd = DomainClassifier(d_model, n_domains, dropout)
        self.grl = GRL(lambda_=1.0)

    def forward(self, x_feat, db_engine_oh, hardware_oh, ram_gb):
        z = self.Gf(x_feat)
        cost_pred = self.Gy(z, db_engine_oh, hardware_oh, ram_gb)
        domain_pred = self.Gd(self.grl(z))
        return cost_pred, domain_pred, z

    @torch.no_grad()
    def predict(self, x_feat, db_engine_oh, hardware_oh, ram_gb):
        """Inference - domain classifier not used"""
        self.eval()
        z = self.Gf(x_feat)
        return self.Gy(z, db_engine_oh, hardware_oh, ram_gb)

print("INFO: DANNCostModel defined")




INFO: GRL defined
INFO: SAINT encoder defined
INFO: Regression head and domain classifier defined
INFO: DANNCostModel defined


In [4]:
# --- Module: eval.py ---
import json
import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr
from torch.utils.data import DataLoader

if False:
    from data import (
        CostModelDataset,
        DOMAIN_COL,
        LABEL_COL,
        QP_EMB_COL,
        expand_qp_emb,
        get_log_scale_cols,
    )
elif False:
    from src.data import (
        CostModelDataset,
        DOMAIN_COL,
        LABEL_COL,
        QP_EMB_COL,
        expand_qp_emb,
        get_log_scale_cols,
    )


def _per_domain_metrics(cost_true_orig, cost_pred_orig, domain):
    per_domain = {}
    for d in sorted(set(domain.tolist())):
        mask = domain == d
        if mask.sum() == 0:
            continue
        yt = cost_true_orig[mask]
        yp = cost_pred_orig[mask]
        rmse = float(np.sqrt(np.mean((yp - yt) ** 2)))
        nz = np.abs(yt) > 1e-8
        mape = float(np.mean(np.abs((yp[nz] - yt[nz]) / yt[nz])) * 100.0) if nz.any() else float("nan")
        rho, _ = spearmanr(yt, yp) if len(yt) > 2 else (float("nan"), 1.0)
        per_domain[str(int(d))] = {
            "n": int(mask.sum()),
            "rmse": rmse,
            "mape": mape,
            "spearman": float(rho),
        }
    return per_domain


def evaluate_on_raw_dataframe(
    model,
    raw_df,
    db_enc,
    hw_enc,
    feat_scaler,
    feat_cols,
    mask_cols,
    qp_cols,
    cost_mu_map,
    cost_sigma_map,
    batch_size=256,
    target_transform=None,
):
    eval_df = raw_df.copy()

    if eval_df[QP_EMB_COL].dtype == object and isinstance(eval_df[QP_EMB_COL].iloc[0], str):
        eval_df[QP_EMB_COL] = eval_df[QP_EMB_COL].apply(
            lambda x: json.loads(x) if isinstance(x, str) else x
        )

    log_scale_cols = get_log_scale_cols(eval_df)
    for col in log_scale_cols:
        eval_df[col] = np.log1p(eval_df[col].clip(lower=0))

    eval_df[feat_cols] = eval_df[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    eval_df[mask_cols] = eval_df[mask_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    for col in feat_cols:
        eval_df[col] = np.clip(eval_df[col], -1e4, 1e4)

    emb_df_eval, _ = expand_qp_emb(eval_df)
    emb_df_eval = emb_df_eval.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    eval_df = pd.concat([eval_df.reset_index(drop=True), emb_df_eval.reset_index(drop=True)], axis=1)

    eval_df[LABEL_COL] = pd.to_numeric(eval_df[LABEL_COL], errors="coerce")
    finite_cost_mask = np.isfinite(eval_df[LABEL_COL].to_numpy())
    if not finite_cost_mask.all():
        eval_df = eval_df.loc[finite_cost_mask].reset_index(drop=True)

    if len(eval_df) == 0:
        return {
            "mse_orig": float("nan"),
            "rmse_orig": float("nan"),
            "mae_orig": float("nan"),
            "nrmse_pct": float("nan"),
            "mape_pct": float("nan"),
            "smape_pct": float("nan"),
            "spearman_rho": float("nan"),
            "spearman_pval": float("nan"),
            "domain_acc": float("nan"),
            "n_samples": 0,
            "pooled": {"rmse": float("nan"), "mape": float("nan"), "spearman": float("nan"), "nrmse": float("nan")},
            "per_domain": {},
        }

    if "__global__" in cost_mu_map:
        global_mu = float(cost_mu_map["__global__"])
        global_sigma = float(max(cost_sigma_map["__global__"], 1e-8))
        mu_eval = pd.Series(global_mu, index=eval_df.index)
        sigma_eval = pd.Series(global_sigma, index=eval_df.index)
    else:
        global_mu = float(np.mean(list(cost_mu_map.values())))
        global_sigma = float(np.mean(list(cost_sigma_map.values())))
        mu_eval = eval_df["db_engine"].map(cost_mu_map).fillna(global_mu)
        sigma_eval = eval_df["db_engine"].map(cost_sigma_map).fillna(global_sigma).clip(lower=1e-8)
    eval_df[LABEL_COL] = (eval_df[LABEL_COL] - mu_eval) / sigma_eval

    finite_norm_mask = np.isfinite(eval_df[LABEL_COL].to_numpy())
    if not finite_norm_mask.all():
        eval_df = eval_df.loc[finite_norm_mask].reset_index(drop=True)

    if len(eval_df) == 0:
        return {
            "mse_orig": float("nan"),
            "rmse_orig": float("nan"),
            "mae_orig": float("nan"),
            "nrmse_pct": float("nan"),
            "mape_pct": float("nan"),
            "smape_pct": float("nan"),
            "spearman_rho": float("nan"),
            "spearman_pval": float("nan"),
            "domain_acc": float("nan"),
            "n_samples": 0,
            "pooled": {"rmse": float("nan"), "mape": float("nan"), "spearman": float("nan"), "nrmse": float("nan")},
            "per_domain": {},
        }

    device = next(model.parameters()).device
    eval_ds = CostModelDataset(eval_df, feat_cols, mask_cols, qp_cols, db_enc, hw_enc, feat_scaler)
    eval_loader = DataLoader(eval_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    model.eval()
    all_cost_pred = []
    all_cost_true = []
    all_domain_pred = []
    all_domain_true = []

    with torch.no_grad():
        for batch in eval_loader:
            x_feat = batch["x_feat"].to(device)
            cost_pred, domain_pred, _ = model(
                x_feat,
                batch["db_oh"].to(device),
                batch["hw_oh"].to(device),
                batch["ram"].to(device),
            )
            all_cost_pred.append(cost_pred.cpu().numpy())
            all_cost_true.append(batch["cost"].numpy())
            all_domain_pred.append(domain_pred.argmax(1).cpu().numpy())
            all_domain_true.append(batch["domain"].numpy())

    cost_pred_norm = np.concatenate(all_cost_pred)
    cost_true_norm = np.concatenate(all_cost_true)
    domain_pred_all = np.concatenate(all_domain_pred)
    domain_true_all = np.concatenate(all_domain_true)

    if "__global__" in cost_mu_map:
        mu_eval_arr = np.full(len(eval_df), float(cost_mu_map["__global__"]), dtype=np.float32)
        sigma_eval_arr = np.full(
            len(eval_df), float(max(cost_sigma_map["__global__"], 1e-8)), dtype=np.float32
        )
    else:
        global_mu = float(np.mean(list(cost_mu_map.values())))
        global_sigma = float(np.mean(list(cost_sigma_map.values())))
        mu_eval_arr = eval_df["db_engine"].map(cost_mu_map).fillna(global_mu).to_numpy(dtype=np.float32)
        sigma_eval_arr = (
            eval_df["db_engine"].map(cost_sigma_map).fillna(global_sigma).clip(lower=1e-8).to_numpy(dtype=np.float32)
        )

    cost_pred_orig = cost_pred_norm * sigma_eval_arr + mu_eval_arr
    cost_true_orig = cost_true_norm * sigma_eval_arr + mu_eval_arr

    if target_transform and target_transform.get("type") == "log1p":
        cost_pred_orig = np.expm1(cost_pred_orig)
        cost_true_orig = np.expm1(cost_true_orig)

    finite_pair_mask = np.isfinite(cost_pred_orig) & np.isfinite(cost_true_orig)
    if not finite_pair_mask.all():
        cost_pred_orig = cost_pred_orig[finite_pair_mask]
        cost_true_orig = cost_true_orig[finite_pair_mask]
        domain_pred_all = domain_pred_all[finite_pair_mask]
        domain_true_all = domain_true_all[finite_pair_mask]

    if len(cost_true_orig) == 0:
        return {
            "mse_orig": float("nan"),
            "rmse_orig": float("nan"),
            "mae_orig": float("nan"),
            "nrmse_pct": float("nan"),
            "mape_pct": float("nan"),
            "smape_pct": float("nan"),
            "spearman_rho": float("nan"),
            "spearman_pval": float("nan"),
            "domain_acc": float("nan"),
            "n_samples": 0,
            "pooled": {"rmse": float("nan"), "mape": float("nan"), "spearman": float("nan"), "nrmse": float("nan")},
            "per_domain": {},
        }

    err = cost_pred_orig - cost_true_orig
    abs_err = np.abs(err)

    mse_orig = float(np.mean(err ** 2))
    rmse_orig = float(np.sqrt(mse_orig))
    mae_orig = float(np.mean(abs_err))

    orig_range = float(np.ptp(cost_true_orig))
    nrmse_pct = float((rmse_orig / (orig_range + 1e-8)) * 100.0)

    mape_mask = np.abs(cost_true_orig) > 1e-8
    mape_pct = (
        float(
            np.mean(
                np.abs((cost_pred_orig[mape_mask] - cost_true_orig[mape_mask]) / cost_true_orig[mape_mask])
            )
            * 100.0
        )
        if mape_mask.any()
        else float("nan")
    )

    smape_pct = float(
        np.mean((2.0 * abs_err) / (np.abs(cost_true_orig) + np.abs(cost_pred_orig) + 1e-8)) * 100.0
    )

    spearman_rho, spearman_pval = spearmanr(cost_true_orig, cost_pred_orig)
    domain_acc = float(np.mean(domain_pred_all == domain_true_all))

    per_domain = _per_domain_metrics(cost_true_orig, cost_pred_orig, domain_true_all)

    return {
        "mse_orig": mse_orig,
        "rmse_orig": rmse_orig,
        "mae_orig": mae_orig,
        "nrmse_pct": nrmse_pct,
        "mape_pct": mape_pct,
        "smape_pct": smape_pct,
        "spearman_rho": float(spearman_rho),
        "spearman_pval": float(spearman_pval),
        "domain_acc": domain_acc,
        "n_samples": int(len(cost_true_orig)),
        "pooled": {
            "rmse": rmse_orig,
            "mape": mape_pct,
            "spearman": float(spearman_rho),
            "nrmse": nrmse_pct,
        },
        "per_domain": per_domain,
    }


print("INFO: evaluate_on_raw_dataframe ready")



INFO: evaluate_on_raw_dataframe ready


In [5]:
# --- Module: training.py ---
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

if False:
    from data import (
        CostModelDataset,
        DOMAIN_COL,
        SOURCE_DOMAIN,
        build_sampler,
        preprocess,
    )
    from eval import evaluate_on_raw_dataframe
    from models import DANNCostModel
elif False:
    from src.data import (
        CostModelDataset,
        DOMAIN_COL,
        SOURCE_DOMAIN,
        build_sampler,
        preprocess,
    )
    from src.eval import evaluate_on_raw_dataframe
    from src.models import DANNCostModel


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def compute_lambda(p, max_lambda=0.1):
    """
    Lambda schedule for large domain shift.

    - Warm up to max_lambda over first 30% of training
    - Hold constant for rest of training
    """
    if p < 0.3:
        return (p / 0.3) * max_lambda
    return max_lambda


print("INFO: compute_lambda defined")


def _domain_count_map(df, col=DOMAIN_COL):
    if col not in df.columns or len(df) == 0:
        return {}
    vc = df[col].value_counts().sort_index()
    return {int(k): int(v) for k, v in vc.items()}


def _format_domain_counts(counts):
    if not counts:
        return "{}"
    return "{" + ", ".join([f"d{k}:{v}" for k, v in counts.items()]) + "}"


def train_epoch(model, loader, optimizer, device, lambda_grl=1.0, alpha_target=0.0, domain_class_weights=None):
    """Train for one epoch."""
    model.train()
    model.grl.set_lambda(lambda_grl)

    total_loss = task_sum = domain_sum = 0.0
    n = 0

    for batch in loader:
        x_feat = batch["x_feat"].to(device)
        db_oh = batch["db_oh"].to(device)
        hw_oh = batch["hw_oh"].to(device)
        ram = batch["ram"].to(device)
        cost = batch["cost"].to(device)
        domain = batch["domain"].to(device)
        has_label = batch["has_label"].to(device)

        cost_pred, domain_pred, _ = model(x_feat, db_oh, hw_oh, ram)
        cost_pred = torch.clamp(cost_pred, min=-100, max=100)

        src_mask = has_label.bool()
        task_loss = (
            F.mse_loss(cost_pred[src_mask], cost[src_mask])
            if src_mask.sum() > 0
            else torch.tensor(0.0, device=device)
        )

        tgt_mask = (~(domain == SOURCE_DOMAIN)) & has_label.bool()
        if alpha_target > 0 and tgt_mask.sum() > 0:
            task_loss = task_loss + alpha_target * F.mse_loss(cost_pred[tgt_mask], cost[tgt_mask])

        domain_loss = F.cross_entropy(domain_pred, domain, weight=domain_class_weights)
        loss = task_loss + lambda_grl * domain_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        task_sum += task_loss.item()
        domain_sum += domain_loss.item()
        n += 1

    if n == 0:
        return {"loss": float("nan"), "task_loss": float("nan"), "domain_loss": float("nan")}

    return {"loss": total_loss / n, "task_loss": task_sum / n, "domain_loss": domain_sum / n}


def evaluate(model, loader, device):
    """Evaluate on labeled samples (for validation)."""
    model.eval()
    all_cp, all_ct, all_dp, all_dt, all_hl = [], [], [], [], []

    with torch.no_grad():
        for batch in loader:
            x_feat = batch["x_feat"].to(device)
            cp, dp, _ = model(
                x_feat,
                batch["db_oh"].to(device),
                batch["hw_oh"].to(device),
                batch["ram"].to(device),
            )

            cp = torch.clamp(cp, min=-100, max=100)

            all_cp.append(cp.cpu())
            all_ct.append(batch["cost"])
            all_dp.append(dp.argmax(1).cpu())
            all_dt.append(batch["domain"])
            all_hl.append(batch["has_label"])

    cp = torch.cat(all_cp)
    ct = torch.cat(all_ct)
    dp = torch.cat(all_dp)
    dt = torch.cat(all_dt)
    hl = torch.cat(all_hl).bool()

    results = {}
    for d in range(4):
        mask = (dt == d) & hl
        if mask.sum() > 0:
            mse_val = F.mse_loss(cp[mask], ct[mask]).item()
            results[f"mse_domain_{d}"] = mse_val

    acc = (dp == dt).float().mean().item()
    results["domain_acc"] = acc
    results["h_divergence_approx"] = max(0.0, 2 * (2 * acc - 1))
    return results


def evaluate_all(model, loader, device, cost_mu_map=None, cost_sigma_map=None, db_classes=None, target_transform=None):
    """Evaluate on all samples (including unlabeled target)."""
    model.eval()
    all_cp, all_ct, all_dt, all_rt, all_db = [], [], [], [], []

    with torch.no_grad():
        for batch in loader:
            x_feat = batch["x_feat"].to(device)
            cp, _, _ = model(
                x_feat,
                batch["db_oh"].to(device),
                batch["hw_oh"].to(device),
                batch["ram"].to(device),
            )

            cp = torch.clamp(cp, min=-100, max=100)

            all_cp.append(cp.cpu().numpy())
            all_ct.append(batch["cost"].numpy())
            all_dt.append(batch["domain"].numpy())
            if "raw_cost" in batch:
                all_rt.append(batch["raw_cost"].numpy())
            all_db.append(batch["db_oh"].argmax(1).numpy())

    cp = np.concatenate(all_cp)
    ct = np.concatenate(all_ct)
    dt = np.concatenate(all_dt)

    if all_rt:
        rt = np.concatenate(all_rt)
        db = np.concatenate(all_db)
        
        cp_orig = np.zeros_like(cp)
        if cost_mu_map and "__global__" in cost_mu_map:
            cp_orig = cp * cost_sigma_map["__global__"] + cost_mu_map["__global__"]
        elif cost_mu_map and db_classes is not None:
            for idx, c in enumerate(db_classes):
                mask = db == idx
                sigma = cost_sigma_map.get(c, 1.0)
                mu = cost_mu_map.get(c, 0.0)
                cp_orig[mask] = cp[mask] * sigma + mu
        else:
            cp_orig = cp.copy()
            
        if target_transform and target_transform.get("type") == "log1p":
            cp_orig = np.expm1(np.maximum(cp_orig, 0.0))
            
        cp_orig = np.clip(cp_orig, 0.0, 1e12)

    results = {}
    for d in range(4):
        mask = dt == d
        if mask.sum() > 0:
            mse_val = float(np.mean((cp[mask] - ct[mask])**2))
            results[f"mse_domain_{d}_all"] = mse_val
            
            if all_rt:
                rt_d = rt[mask]
                cp_d = cp_orig[mask]
                rmse = float(np.sqrt(np.mean((rt_d - cp_d)**2)))
                nz = np.abs(rt_d) > 1e-8
                mape = float(np.mean(np.abs((cp_d[nz] - rt_d[nz]) / rt_d[nz])) * 100.0) if nz.any() else float("nan")
                results[f"rmse_domain_{d}_all"] = rmse
                results[f"mape_domain_{d}_all"] = mape

    return results


print("INFO: Training helpers defined")


def train(
    da,
    n_epochs=50,
    batch_size=256,
    lr=1e-3,
    d_model=128,
    n_heads=4,
    n_layers=2,
    dropout=0.1,
    alpha_target=0.0,
    labeled_target_frac=0.0,
    normalization_mode="global",
    log_target=False,
    lambda_max=0.1,
    use_sampler=False,
    seed=42,
    device_str="cuda",
):
    device = torch.device("cuda" if torch.cuda.is_available() and device_str == "cuda" else "cpu")
    set_seed(seed)

    (
        df,
        feat_cols,
        mask_cols,
        qp_cols,
        db_enc,
        hw_enc,
        feat_scaler,
        cost_mu_map,
        cost_sigma_map,
        target_transform,
    ) = preprocess(da, normalization_mode=normalization_mode, log_target=log_target)

    train_idx, val_idx = train_test_split(
        np.arange(len(df)),
        test_size=0.15,
        stratify=df[DOMAIN_COL].values,
        random_state=seed,
    )

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)

    split_sizes = {"train": len(train_df), "val": len(val_df)}
    split_domain_counts = {
        "train": _domain_count_map(train_df),
        "val": _domain_count_map(val_df),
    }

    labeled_target_idx = None
    if labeled_target_frac > 0:
        rng = np.random.RandomState(seed)
        tgt_idx = np.where(train_df[DOMAIN_COL].values != SOURCE_DOMAIN)[0]
        if len(tgt_idx) > 0:
            n_label = max(1, int(len(tgt_idx) * labeled_target_frac))
            labeled_target_idx = rng.choice(tgt_idx, size=n_label, replace=False).tolist()

    ds_kw = dict(
        feat_cols=feat_cols,
        mask_cols=mask_cols,
        qp_cols=qp_cols,
        db_enc=db_enc,
        hw_enc=hw_enc,
        feat_scaler=feat_scaler,
    )

    train_ds = CostModelDataset(train_df, **ds_kw, labeled_target_idx=labeled_target_idx, 
                                raw_costs=train_df["raw_cost"].values if "raw_cost" in train_df else None)
    val_ds = CostModelDataset(val_df, **ds_kw, 
                              raw_costs=val_df["raw_cost"].values if "raw_cost" in val_df else None)

    class_counts = np.bincount(train_ds.domain, minlength=4).astype(np.float32)
    class_counts = np.maximum(class_counts, 1.0)
    domain_class_weights = (len(train_ds) / (4.0 * class_counts)).astype(np.float32)
    domain_class_weights = domain_class_weights / domain_class_weights.mean()
    domain_class_weights = torch.tensor(domain_class_weights, dtype=torch.float32, device=device)

    if use_sampler:
        sampler = build_sampler(train_ds)
        train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=0)
    else:
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)

    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    num_feat_total = len(feat_cols) + len(mask_cols) + len(qp_cols)
    model = DANNCostModel(
        num_features=num_feat_total,
        n_db_engines=len(db_enc.classes_),
        n_hardware=len(hw_enc.classes_),
        d_model=d_model,
        n_heads=n_heads,
        n_layers=n_layers,
        n_domains=4,
        dropout=dropout,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=1e-5)

    best_mse = float("inf")
    best_state = None
    best_epoch = 0
    total_steps = n_epochs * max(len(train_loader), 1)
    step = 0
    history = []
    start_time = time.time()

    for epoch in range(1, n_epochs + 1):
        lam = compute_lambda(step / max(total_steps, 1), max_lambda=lambda_max)
        tr = train_epoch(
            model,
            train_loader,
            optimizer,
            device,
            lambda_grl=lam,
            alpha_target=alpha_target,
            domain_class_weights=domain_class_weights,
        )
        step += len(train_loader)
        scheduler.step()

        val = evaluate(model, val_loader, device)
        val_all = evaluate_all(model, val_loader, device, 
                               cost_mu_map=cost_mu_map, 
                               cost_sigma_map=cost_sigma_map, 
                               db_classes=db_enc.classes_, 
                               target_transform=target_transform)

        src_mse = val.get("mse_domain_3", float("nan"))
        tgt_mse = np.nanmean(
            [val_all.get(f"mse_domain_{d}_all", float("nan")) for d in [0, 1, 2]]
        )
        
        src_rmse = val_all.get("rmse_domain_3_all", float("nan"))
        src_mape = val_all.get("mape_domain_3_all", float("nan"))
        tgt_rmse = np.nanmean([val_all.get(f"rmse_domain_{d}_all", float("nan")) for d in [0, 1, 2]])
        tgt_mape = np.nanmean([val_all.get(f"mape_domain_{d}_all", float("nan")) for d in [0, 1, 2]])
        
        # Add print statement to show progress
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:02d}/{n_epochs} | "
                  f"TrLoss: {tr['loss']:.2f} | "
                  f"SrcMSE(Norm): {src_mse:.3f} | TgtMSE(Norm): {tgt_mse:.3f} | "
                  f"SrcRMSE: {src_rmse:.0f} (MAPE: {src_mape:.1f}%) | "
                  f"TgtRMSE: {tgt_rmse:.0f} (MAPE: {tgt_mape:.1f}%) | "
                  f"DomAcc: {val['domain_acc']:.3f}")

        history.append(
            dict(
                epoch=epoch,
                lambda_grl=lam,
                train_loss=tr["loss"],
                task_loss=tr["task_loss"],
                domain_loss=tr["domain_loss"],
                src_mse=src_mse,
                tgt_mse=tgt_mse,
                domain_acc=val["domain_acc"],
                h_div=val["h_divergence_approx"],
            )
        )

        if src_mse < best_mse:
            best_mse = src_mse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch

    if best_state is not None:
        model.load_state_dict(best_state)

    train_minutes = (time.time() - start_time) / 60.0
    history_df = pd.DataFrame(history)

    return {
        "model": model,
        "db_enc": db_enc,
        "hw_enc": hw_enc,
        "feat_scaler": feat_scaler,
        "feat_cols": feat_cols,
        "mask_cols": mask_cols,
        "qp_cols": qp_cols,
        "history_df": history_df,
        "cost_mu_map": cost_mu_map,
        "cost_sigma_map": cost_sigma_map,
        "target_transform": target_transform,
        "best_epoch": best_epoch,
        "train_minutes": train_minutes,
        "split_sizes": split_sizes,
        "split_domain_counts": split_domain_counts,
    }


print("INFO: Main training function defined")


def train_dann_custom(
    da,
    n_epochs=30,
    batch_size=256,
    lr=1e-3,
    alpha_target=0.0,
    labeled_target_frac=0.0,
    normalization_mode="global",
    log_target=False,
    seed=42,
    device_str="cuda",
    lambda_max=0.1,
    use_sampler=False,
):
    return train(
        da,
        n_epochs=n_epochs,
        batch_size=batch_size,
        lr=lr,
        alpha_target=alpha_target,
        labeled_target_frac=labeled_target_frac,
        normalization_mode=normalization_mode,
        log_target=log_target,
        lambda_max=lambda_max,
        use_sampler=use_sampler,
        seed=seed,
        device_str=device_str,
    )


def train_no_dann_supervised(
    da,
    n_epochs=30,
    batch_size=256,
    lr=1e-3,
    normalization_mode="global",
    log_target=False,
    seed=42,
    device_str="cuda",
):
    return train(
        da,
        n_epochs=n_epochs,
        batch_size=batch_size,
        lr=lr,
        alpha_target=0.0,
        labeled_target_frac=1.0,
        normalization_mode=normalization_mode,
        log_target=log_target,
        lambda_max=0.0,
        use_sampler=False,
        seed=seed,
        device_str=device_str,
    )


def train_and_eval(
    da,
    normalization_mode="global",
    n_epochs=30,
    batch_size=256,
    lr=1e-3,
    seed=42,
    device_str="cuda",
    log_target=False,
    use_sampler=False,
):
    da_train_val, da_test_raw = train_test_split(
        da,
        test_size=0.15,
        stratify=da[DOMAIN_COL].values,
        random_state=seed,
    )

    out = train_dann_custom(
        da_train_val,
        n_epochs=n_epochs,
        batch_size=batch_size,
        lr=lr,
        alpha_target=0.0,
        labeled_target_frac=0.0,
        normalization_mode=normalization_mode,
        log_target=log_target,
        seed=seed,
        device_str=device_str,
        lambda_max=0.1,
        use_sampler=use_sampler,
    )

    eval_metrics = evaluate_on_raw_dataframe(
        out["model"],
        da_test_raw,
        out["db_enc"],
        out["hw_enc"],
        out["feat_scaler"],
        out["feat_cols"],
        out["mask_cols"],
        out["qp_cols"],
        out["cost_mu_map"],
        out["cost_sigma_map"],
        batch_size=batch_size,
        target_transform=out["target_transform"],
    )

    metrics = {
        "pooled": eval_metrics["pooled"],
        "per_domain": eval_metrics["per_domain"],
        "domain_acc": eval_metrics["domain_acc"],
        "best_epoch": out["best_epoch"],
        "train_minutes": out["train_minutes"],
    }

    return metrics, out["history_df"], out["model"], eval_metrics["per_domain"]


def run_ablation_suite(
    da,
    normalization_mode="global",
    n_epochs=30,
    batch_size=256,
    lr=1e-3,
    seed=42,
    device_str="cuda",
    log_target=False,
):
    da_train_val, da_test_raw = train_test_split(
        da, test_size=0.15, stratify=da[DOMAIN_COL].values, random_state=seed
    )

    trainval_counts = _domain_count_map(da_train_val)
    test_counts = _domain_count_map(da_test_raw)
    print(f"Global split sizes -> train_val={len(da_train_val)}, test={len(da_test_raw)}")
    print(f"Domain counts train_val: {_format_domain_counts(trainval_counts)}")
    print(f"Domain counts test:      {_format_domain_counts(test_counts)}")

    configs = [
        {"name": "A_dann_source_only", "kind": "dann", "alpha_target": 0.0, "labeled_target_frac": 0.0},
        {"name": "B_dann_ssda_10pct", "kind": "dann", "alpha_target": 0.3, "labeled_target_frac": 0.10},
        {"name": "C_no_dann_supervised", "kind": "no_dann"},
    ]

    rows = []
    artifacts = {}

    for cfg in configs:
        print("=" * 90)
        print(f"Running {cfg['name']} | normalization={normalization_mode}")
        print("=" * 90)

        if cfg["kind"] == "dann":
            out = train_dann_custom(
                da_train_val,
                n_epochs=n_epochs,
                batch_size=batch_size,
                lr=lr,
                alpha_target=cfg["alpha_target"],
                labeled_target_frac=cfg["labeled_target_frac"],
                normalization_mode=normalization_mode,
                log_target=log_target,
                seed=seed,
                device_str=device_str,
            )
        else:
            out = train_no_dann_supervised(
                da_train_val,
                n_epochs=n_epochs,
                batch_size=batch_size,
                lr=lr,
                normalization_mode=normalization_mode,
                log_target=log_target,
                seed=seed,
                device_str=device_str,
            )

        split_sizes = out.get("split_sizes", {})
        split_domain_counts = out.get("split_domain_counts", {})

        tr_n = split_sizes.get("train", "?")
        va_n = split_sizes.get("val", "?")
        te_n = len(da_test_raw)
        print(f"Split sizes ({cfg['name']}) -> train={tr_n}, val={va_n}, test={te_n}")

        if split_domain_counts:
            tr_counts = split_domain_counts.get("train", {})
            va_counts = split_domain_counts.get("val", {})
            print(f"Domain counts train ({cfg['name']}): {_format_domain_counts(tr_counts)}")
            print(f"Domain counts val   ({cfg['name']}): {_format_domain_counts(va_counts)}")
        else:
            print(f"Domain counts test  ({cfg['name']}): {_format_domain_counts(test_counts)}")

        metrics = evaluate_on_raw_dataframe(
            out["model"],
            da_test_raw,
            out["db_enc"],
            out["hw_enc"],
            out["feat_scaler"],
            out["feat_cols"],
            out["mask_cols"],
            out["qp_cols"],
            out["cost_mu_map"],
            out["cost_sigma_map"],
            batch_size=batch_size,
            target_transform=out["target_transform"],
        )

        row = {"experiment": cfg["name"], **metrics}
        rows.append(row)
        artifacts[cfg["name"]] = out

        print(
            f"{cfg['name']} -> RMSE={metrics['rmse_orig']:.4f}, "
            f"NRMSE={metrics['nrmse_pct']:.2f}%, MAPE={metrics['mape_pct']:.2f}%, "
            f"Spearman={metrics['spearman_rho']:.4f}, DomainAcc={metrics['domain_acc']:.4f}"
        )

    results_df = pd.DataFrame(rows).sort_values("rmse_orig").reset_index(drop=True)
    print("\nA/B/C Comparison (sorted by RMSE):")
    print(results_df.to_string(index=False))

    return results_df, artifacts


print("INFO: run_ablation_suite ready")



INFO: compute_lambda defined
INFO: Training helpers defined
INFO: Main training function defined
INFO: run_ablation_suite ready


In [6]:
import json, os, hashlib, random
import numpy as np
import torch
import pandas as pd

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

SEED = 42
cfg = {"seed": SEED, "n_epochs": 30, "batch_size": 256, "lr": 1e-3, "normalization_mode": "global", "log_target": False, "use_sampler": False}
cfg_hash = hashlib.sha1(json.dumps(cfg, sort_keys=True).encode()).hexdigest()[:8]
RUN_TAG = f"baseline_{cfg_hash}_{SEED}"
print(f"Starting run {RUN_TAG}")

set_seed(SEED)

try:
    df = pd.read_csv('/kaggle/input/olap-dataset-dnn-03/final_combined_olap.csv')
except FileNotFoundError:
    df = pd.read_csv('/kaggle/input/datasets/phmnmendis/olap-dataset-dnn-03/final_combined_olap.csv')

print(f'Data shape: {df.shape}')

metrics, history_df, model, per_domain = train_and_eval(
    df,
    normalization_mode=cfg['normalization_mode'],
    n_epochs=cfg['n_epochs'],
    batch_size=cfg['batch_size'],
    lr=cfg['lr'],
    seed=cfg['seed'],
    device_str='cuda' if torch.cuda.is_available() else 'cpu',
    log_target=cfg['log_target'],
    use_sampler=cfg['use_sampler'],
)

os.makedirs('/kaggle/working/artifacts', exist_ok=True)
with open('/kaggle/working/artifacts/test_metrics.json', 'w') as f:
    json.dump({
        'run_tag': RUN_TAG,
        'cfg_hash': cfg_hash,
        'seed': SEED,
        'pooled': metrics['pooled'],
        'per_domain': metrics['per_domain'],
        'domain_acc': metrics['domain_acc'],
        'best_epoch': metrics['best_epoch'],
        'train_minutes': metrics['train_minutes'],
    }, f, indent=2)

history_df.to_csv('/kaggle/working/artifacts/history.csv', index=False)
torch.save({'state_dict': model.state_dict(), 'cfg': cfg, 'cfg_hash': cfg_hash}, '/kaggle/working/artifacts/best.pt')
with open('/kaggle/working/artifacts/meta.json', 'w') as f:
    json.dump(cfg, f, indent=2, default=str)

print(f'DONE - run_tag={RUN_TAG} cfg_hash={cfg_hash}')



Starting run baseline_70ca141e_42
Data shape: (10320, 125)
INFO: Converted qp_emb_vector from JSON strings
INFO: Log-scaling 29 feature columns
WARN: Dropped 6 zero-variance features: ['conflicts', 'read_write_ratio', 'tup_deleted', 'tup_inserted', 'tup_updated']
INFO: Expanded 384 query plan embedding dimensions
WARN: Dropped 2 rows with non-finite cost values before normalization
INFO: Fitted encoders: 2 DB engines, 2 hardware configs
INFO: Fitting scaler on 5932 source domain samples

INFO: Cost normalization mode: GLOBAL
   mu=5.1764, sigma=8.8434
   Normalized overall: mu=-0.0000, sigma=1.0000
   Range: [-0.6984, 12.1607]
   Normalized MySQL domains (0,1): mu=1.1342, sigma=1.7231
   Normalized PostgreSQL domains (2,3): mu=-0.2945, sigma=0.2610
   Per-engine normalized mean/std:
              mean     std
db_engine                 
mysql       1.1342  1.7231
postgresql -0.2945  0.2610
Epoch 01/30 | TrLoss: 0.07 | SrcMSE(Norm): 0.048 | TgtMSE(Norm): 2.763 | SrcRMSE: 2 (MAPE: 176.6%)